In [1]:
from typing import TypedDict


class FormState(TypedDict):
    age: int | None


from langgraph.types import interrupt


def get_age_node(state: FormState) -> FormState:
    prompt = "请输入你的年龄"
    while True:
        answer = interrupt(prompt)
        if isinstance(answer, int) and answer > 0:
            return {"age": answer}

        prompt = f"输入值 {answer} @ {type(answer)} 无效"


from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

builder = StateGraph(FormState)
builder.add_node("get_age", get_age_node)
builder.add_edge(START, "get_age")
builder.add_edge("get_age", END)
graph = builder.compile(InMemorySaver())

config = {"configurable": {"thread_id": "test"}}
s0 = graph.invoke({"aget": None}, config=config)
print(s0)
from langgraph.types import Command

s1 = graph.invoke(Command(resume="三十"), config=config)
print(s1)

s2 = graph.invoke(Command(resume=-10), config=config)
print(s2)

s3 = graph.invoke(Command(resume=30), config=config)
print(s3)

{'__interrupt__': [Interrupt(value='请输入你的年龄', id='94f59d62c7789617eb57dae9dae3b1ab')]}
{'__interrupt__': [Interrupt(value="输入值 三十 @ <class 'str'> 无效", id='94f59d62c7789617eb57dae9dae3b1ab')]}
{'__interrupt__': [Interrupt(value="输入值 -10 @ <class 'int'> 无效", id='94f59d62c7789617eb57dae9dae3b1ab')]}
{'age': 30}
